In [ ]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load
model = load_model('../model/bilstm_sentiment_v1.keras')
with open('../tokenizer/tokenizer_v1.pkl', 'rb') as f:
    tokenizer = pickle.load(f)


max_len = 50  

In [2]:
hard_cases = pd.DataFrame({
    'cleaned_text': [
        'could not be happier',
        'cannot be more pleased',
        'never been better',
        'not a single complaint',
        'could not ask for more',
        'would not change a thing',
        'not bad at all',
        'never disappoints',
        'could not love this more',
        'not a problem at all',
        'not happy at all',
        'never felt worse',
        'could not be more disappointed',
        'not good enough',
        'not what i expected',
        'cannot stand this',
        'not great not terrible',
        'not the best not the worst',
        'neither good nor bad',
        'not sure how i feel about this',
    ],
    'sentiment': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                  0, 0, 0, 0, 0, 0, 2, 2, 2, 2]
})

In [ ]:
extra_hard_cases = pd.DataFrame({
    'cleaned_text': [
        # never been so + positive emotion
        'never been so relieved',
        'never been so excited',
        'never been so grateful',
        'never been so happy',
        'never been so pleased',
        'never been so impressed',
        'never felt so good',
        'never felt so relieved',
        'never felt so grateful',

        # never been so + negative emotion
        'never been so frustrated',
        'never been so disappointed',
        'never been so upset',
        'never felt so bad',
        'never felt so let down',

        # could not be more + positive
        'could not be more excited',
        'could not be more grateful',
        'could not be more relieved',
        'could not be more impressed',
        'could not be more satisfied',

        # could not be more + negative
        'could not be more frustrated',
        'could not be more upset',
        'could not be more disappointed',
        'could not be more annoyed',

        # model thinks these are negative but they are positive
        'utterly fantastic',
        'blown away honestly', 
        'its alright nothing special',
        'utterly brilliant',
        'utterly amazing',
        'blown away by this',
        'completely blown away',
        'utterly speechless',
        'honestly blown away',
        'alright i guess nothing special',
        'its alright',
        'alright nothing more',
        'pretty alright overall',
        'alright but nothing special',
        'alright enough',

        'does the job nothing more',
        'neither impressed nor disappointed',
        'neither good nor bad',
        'does what it needs to nothing more',
        'gets the job done nothing special',

        #understated negative
        'i would not exactly call this a success',
        'could have gone better if i am being honest',
        'not sure this was the best decision i have made',
        'lets just say it did not go as planned',
        'not quite what i had in mind',
        'would not call this a win exactly',
        'not exactly my finest moment',
        'could have been handled better',
        'not the outcome i was hoping for',
        'would not say this went well',
        'not ideal if i am being honest',
        'safe to say this did not work out',
        'not exactly a great result',
        'i would not call this a success by any measure',
        'this did not go the way i wanted',
        'not my best decision honestly',
        'could have done without this experience',
        'not what i was hoping for at all',
        'would not recommend based on my experience',
        'not sure i would do this again',

        #hedged_positives
        'not going to lie this is actually pretty good',
        'okay i will admit this is better than i thought',
        'against all odds this actually worked out well',
        'i have to admit this is surprisingly good',
        'not going to lie i actually enjoyed this',
        'okay fine i will admit it was good',
        'surprisingly this actually turned out well',
        'i will be honest this is better than expected',
        'have to say this actually impressed me',
        'not going to lie this exceeded my expectations',
        'okay i admit i was wrong this is great',
        'against my expectations this was really good',
        'i did not expect to like this but i genuinely do',
        'have to be honest this actually worked really well',
        'fine i will say it this is actually good',

        #backhanded neutral
        'for what it costs i guess it is okay',
        'better than i expected which was not much',
        'at least it is not as bad as last time',
        'not terrible for the price i suppose',
        'better than nothing i guess',
        'at least it works most of the time',
        'for the price it is acceptable i suppose',
        'not as bad as i feared honestly',
        'at least it is an improvement on before',
        'better than the last one which is not saying much',

    ],
    'sentiment': [
        1,1,1,1,1,1,1,1,1,  # positive
        0,0,0,0,0,           # negative
        1,1,1,1,1,           # positive
        0,0,0,0,             # negative
        1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2,
        2,2,2,2,2,           #neutral
        0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
        1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
        2,2,2,2,2,2,2,2,2,2,
    ]
})

In [4]:
# Repeat hard cases 80-100x so they don't drown in your 105k dataset
hard_cases_upsampled = pd.concat([hard_cases] * 100, ignore_index=True)

In [5]:
# Load original data
df_original = pd.read_csv('../clean_data/re_clean_dataset.csv')
df_original['cleaned_text'] = df_original['cleaned_text'].fillna('').astype(str)
df_original = df_original[df_original['cleaned_text'].str.strip() != ''].reset_index(drop=True)

# Sample
df_sample = df_original.groupby('sentiment').sample(n=2000, random_state=42)

# Combine old hard cases + new extra hard cases
all_hard_cases = pd.concat([hard_cases, extra_hard_cases], ignore_index=True)
hard_cases_upsampled = pd.concat([all_hard_cases] * 150, ignore_index=True)

# Combine everything
df_finetune = pd.concat([df_sample, hard_cases_upsampled], ignore_index=True)
df_finetune = df_finetune.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_finetune['sentiment'].value_counts())

sentiment
1.0    9050
0.0    7250
2.0    5900
Name: count, dtype: int64


In [6]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
# Always do this before tokenizing
df_finetune['cleaned_text'] = df_finetune['cleaned_text'].fillna('').astype(str)
df_finetune = df_finetune[df_finetune['cleaned_text'].str.strip() != ''].reset_index(drop=True)

X = df_finetune['cleaned_text'].values
Y = df_finetune['sentiment'].values

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')
Y_cat = to_categorical(Y, num_classes=3)

In [7]:
# Clean before tokenizing
df_finetune['text'] = df_finetune['text'].fillna('').astype(str)

X = df_finetune['cleaned_text'].values
Y = df_finetune['sentiment'].values

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')
Y_cat = to_categorical(Y, num_classes=3)

# Split
X_train, X_val, Y_train, Y_val = train_test_split(
    X_pad, Y_cat,
    test_size=0.15,
    stratify=Y,
    random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (18870, 50)
Val: (3330, 50)


In [ ]:
from tensorflow.keras.optimizers import Adam

# Freeze embedding to not overwrite FastText knowledge
model.layers[0].trainable = False

# Recompile with much lower learning rate
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=1e-4),  # 10x lower than original
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 300)        │     6,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 50, 300)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 128)        │       186,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,191,107 (23.62 MB)

 Trainable params: 191,107 (746.51 KB)

 Non-trainable params: 6,000,000 (22.89 MB)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.6183 - loss: 0.9394 - val_accuracy: 0.7240 - val_loss: 0.5623 - learning_rate: 1.0000e-04
Epoch 2/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7676 - loss: 0.4989 - val_accuracy: 0.8372 - val_loss: 0.3553 - learning_rate: 1.0000e-04
Epoch 3/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8673 - loss: 0.3521 - val_accuracy: 0.9243 - val_loss: 0.2337 - learning_rate: 1.0000e-04
Epoch 4/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9279 - loss: 0.2488 - val_accuracy: 0.9559 - val_loss: 0.1581 - learning_rate: 1.0000e-04
Epoch 5/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9478 - loss: 0.1875 - val_accuracy: 0.9610 - val_loss: 0.1277 - learning_rate: 1.0000e-04
Epoch 6/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9534 - loss: 0.1588 - val_accuracy: 0.9604 - val_loss: 0.1158 - learning_rate: 1.0000e-04
Epoch 7/15
590/590 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - acc

In [10]:
def predict(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    pred = model.predict(pad, verbose=0)[0]
    label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}
    print(f"Text     : {text}")
    print(f"Negative : {pred[0]:.3f}")
    print(f"Positive : {pred[1]:.3f}")
    print(f"Neutral  : {pred[2]:.3f}")
    print(f"Result   : {label_map[np.argmax(pred)]}\n")

# Test the exact sentences that were failing before
predict("could not be happier")
predict("not bad at all")
predict("never been better")
predict("not happy at all")       # should still be negative
predict("not great not terrible") # should be neutral

Text     : could not be happier
Negative : 0.001
Positive : 0.999
Neutral  : 0.000
Result   : positive

Text     : not bad at all
Negative : 0.001
Positive : 0.999
Neutral  : 0.000
Result   : positive

Text     : never been better
Negative : 0.008
Positive : 0.991
Neutral  : 0.000
Result   : positive

Text     : not happy at all
Negative : 0.998
Positive : 0.002
Neutral  : 0.000
Result   : negative

Text     : not great not terrible
Negative : 0.000
Positive : 0.000
Neutral  : 1.000
Result   : neutral



In [ ]:
import pickle

model.save('../model/bilstm_sentiment_v6.keras')

with open('../tokenizer/tokenizer_v6.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

import json
config = {
    'max_len': max_len,
    'vocab_size': "same as v1",
    'embedding_dim': "same as v1",
    'version': 'v6',
    'notes': 'Fine tune adding extra hard cases+ neutral values + understated negative + backhanded neutral + hedged positives'
}
with open('config_v6.json', 'w') as f:
    json.dump(config, f, indent=2)

print("v6 saved.")

v6 saved.


In [12]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_val, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(Y_val, axis=1)

print(classification_report(
    y_true, y_pred_classes,
    target_names=['Negative', 'Positive', 'Neutral']
))

# Save v2 only if better than v1
model.save('bilstm_sentiment_v2.keras')
print("v2 saved.")

              precision    recall  f1-score   support

    Negative       0.96      0.94      0.95      1088
    Positive       0.95      0.96      0.96      1357
     Neutral       0.98      0.98      0.98       885

    accuracy                           0.96      3330
   macro avg       0.96      0.96      0.96      3330
weighted avg       0.96      0.96      0.96      3330

v2 saved.


In [13]:
y_pred = model.predict(X_val)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(Y_val, axis=1)

from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(y_true, y_pred_classes)
print("Test Accuracy:", test_accuracy)

105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Test Accuracy: 0.960960960960961
